# 🦾 Accessible Services Navigator: Full System Demo

## Making Nairobi More Accessible with AI Agents

---

### 📋 **What This Notebook Demonstrates**

This comprehensive demo showcases a **production-ready multi-agent system** built with Google's Agent Development Kit (ADK) that helps people with disabilities find accessible healthcare and social services in Nairobi, Kenya.

**The Challenge:**  
People with disabilities face significant barriers accessing services:
- **Limited information** about facility accessibility features
- **Poor geographic coverage** in some areas
- **Cost barriers** with unclear pricing
- **Transportation challenges** without clear guidance

**Our Solution:**  
An intelligent 4-agent system that:
1. Understands user needs (disability type, location, budget)
2. Searches 25+ real facilities with detailed accessibility data
3. Scores and ranks facilities using multiple criteria
4. Generates personalized service plans with travel guidance

**Key Technologies:**
- **Google ADK**: Agent orchestration and memory management
- **Gemini 2.5 Flash Lite**: Fast, cost-effective LLM
- **Pydantic**: Strict data validation and type safety
- **Structured Logging**: Full observability and debugging

---

### 🎬 **What You'll Learn**

By the end of this notebook, you'll understand:
- ✅ Multi-agent architecture with data handoffs
- ✅ Custom tool creation for specialized tasks
- ✅ Memory management for session persistence
- ✅ Orchestration patterns for complex workflows
- ✅ Evaluation methodology for agent systems

Let's dive in! 🚀

## 🏗️ System Architecture

Our multi-agent system follows a **pipeline architecture** where each agent specializes in one task:

```
User Query
    ↓
[Intake Agent] ────→ UserProfile (structured)
    ↓
[Search Agent] ────→ CandidateFacilities (filtered)
    ↓
[Reasoning Agent] ──→ ScoredFacilities (ranked)
    ↓
[Recommendation Agent] ──→ ServicePlan (personalized)
    ↓
User receives complete service plan
```

###  **Data Flow Details**

| **Agent** | **Input** | **Output** | **Responsibility** |
|-----------|-----------|------------|-------------------|
| **Intake** | Natural language query | `UserProfile` | Extract disability type, location, budget, services |
| **Search** | `UserProfile` | `CandidateFacilities` | Filter facilities by criteria using dataset |
| **Reasoning** | `CandidateFacilities` | `ScoredFacilities` | Score facilities on accessibility & relevance |
| **Recommendation** | `ScoredFacilities` | `ServicePlan` | Generate personalized plan with travel guidance |

### 🔑 **Key Design Principles**

1. **Separation of Concerns**: Each agent has one clear responsibility
2. **Type Safety**: All data handoffs use Pydantic models for validation
3. **Debuggability**: Each step's output can be inspected independently
4. **Extensibility**: New agents or tools can be added without breaking existing flow
5. **Memory**: Session and long-term storage for user profiles and history

---

## 📦 Setup: Environment and Imports

First, let's set up our environment and import all necessary modules.

In [ ]:
# Add parent directory to Python path
import sys
import os
from pathlib import Path

# Navigate to implementation directory
if 'implementation' not in os.getcwd():
    os.chdir('..')
sys.path.insert(0, str(Path.cwd()))

print(f"✅ Working directory: {os.getcwd()}")
print(f"✅ Python path configured")

In [ ]:
# Core imports
import asyncio
import json
from datetime import datetime
from typing import Dict, List, Any

# Data handling
import pandas as pd

# Our custom modules
from src.orchestrator import AgentOrchestrator
from src.memory import MemoryManager
from src.models import UserProfile, ServicePlan, Facility

print("✅ All imports successful!")

## 📊 Dataset Preview

Let's quickly explore the facilities dataset to understand what we're working with.

In [ ]:
# Load datasets
clinics_df = pd.read_csv('data/nairobi_clinics_accessible.csv')
social_df = pd.read_csv('data/nairobi_social_services_accessible.csv')

# Combine datasets
all_facilities = pd.concat([clinics_df, social_df], ignore_index=True)

print(f"📊 Total Facilities: {len(all_facilities)}")
print(f"   🏥 Clinics: {len(clinics_df)}")
print(f"   🏢 Social Services: {len(social_df)}")
print(f"\n📍 Coverage: {all_facilities['subcounty'].nunique()} subcounties")
print(f"♿ Accessibility Features: {all_facilities['accessibility_features'].str.split(',').str.len().mean():.1f} per facility (avg)")

# Show sample facility
print("\n" + "="*60)
print("🏥 SAMPLE FACILITY")
print("="*60)
sample = all_facilities.iloc[0]
for field in ['name', 'subcounty', 'ward', 'category', 'disability_support']:
    print(f"{field.upper()}: {sample[field]}")
print(f"ACCESSIBILITY FEATURES: {sample['accessibility_features'][:100]}...")

## 🚀 Initialize the System

Now let's create our orchestrator, which manages all 4 agents and coordinates their work.

In [ ]:
# Initialize orchestrator (creates all 4 agents internally)
orchestrator = AgentOrchestrator()

print("✅ Orchestrator initialized successfully!")
print("\n🤖 Agents Created:")
print("   1️⃣  Intake Agent - Extracts user needs")
print("   2️⃣  Search Agent - Finds candidate facilities")
print("   3️⃣  Reasoning Agent - Scores and ranks facilities")
print("   4️⃣  Recommendation Agent - Generates personalized plans")
print("\n💾 Memory Manager: Ready")
print("📝 Logging: Configured")

## 🎭 Demo Scenario: Wheelchair User in Embakasi

**Meet Sarah:**
- 28-year-old wheelchair user
- Lives in Embakasi East, Nairobi
- Needs affordable healthcare with wheelchair access
- Budget: Low-income (seeking free/subsidized services)

**Her Query:**  
> *"I use a wheelchair and need a clinic in Embakasi East. I can't afford expensive services, so I need something free or cheap. The place must have ramps."*

Let's see how our 4-agent system handles this real-world query! 👇

In [ ]:
# Define our demo query
demo_query = """I use a wheelchair and need a clinic in Embakasi East. 
I can't afford expensive services, so I need something free or cheap. 
The place must have ramps."""

# Create a unique user ID for this demo
user_id = "demo_sarah_wheelchair_user"

print("📝 Query:", demo_query)
print(f"👤 User ID: {user_id}")
print("\n⏳ Processing query through 4-agent pipeline...")
print("="*60)

In [ ]:
# Run the complete pipeline
result = await orchestrator.process_query(
    user_id=user_id,
    query=demo_query
)

print("\n✅ Processing complete!")
print(f"⏱️  Total time: {result.get('total_time_seconds', 0):.2f} seconds")

## 🔍 Agent 1: Intake Agent Output

The **Intake Agent** extracted structured data from Sarah's natural language query.

In [ ]:
# Display Intake Agent output
if 'user_profile' in result:
    profile = result['user_profile']
    print("🎯 EXTRACTED USER PROFILE")
    print("="*60)
    print(f"♿ Disability Type: {profile.get('disability_type', 'N/A')}")
    print(f"📍 Preferred Location: {profile.get('preferred_location', 'N/A')}")
    print(f"💰 Budget: {profile.get('budget', 'N/A')}")
    print(f"🏥 Service Needed: {profile.get('service_needed', 'N/A')}")
    print(f"✅ Required Features: {', '.join(profile.get('required_features', []))}")
    print("\n💡 Key Achievement: Natural language → Structured data")
    print(f"⏱️  Time taken: {result.get('intake_time_seconds', 0):.2f}s")
else:
    print("⚠️ No user profile extracted")

## 🔎 Agent 2: Search Agent Output

The **Search Agent** used the UserProfile to filter our dataset and find matching facilities.

In [ ]:
# Display Search Agent output
if 'candidate_facilities' in result:
    candidates = result['candidate_facilities']
    count = len(candidates)
    print(f"🔍 FOUND {count} CANDIDATE FACILITIES")
    print("="*60)
    
    if count > 0:
        print("\n📋 List of Candidates:")
        for i, facility in enumerate(candidates[:5], 1):  # Show first 5
            print(f"\n{i}. {facility['name']}")
            print(f"   📍 Location: {facility.get('subcounty', 'N/A')}, {facility.get('ward', 'N/A')}")
            print(f"   ♿ Supports: {facility.get('disability_support', 'N/A')}")
            print(f"   💰 Cost: {facility.get('cost_estimate', 'N/A')}")
        
        if count > 5:
            print(f"\n   ... and {count - 5} more facilities")
        
        print(f"\n💡 Key Achievement: Filtered {len(all_facilities)} facilities → {count} matches")
        print(f"⏱️  Time taken: {result.get('search_time_seconds', 0):.2f}s")
    else:
        print("⚠️ No matching facilities found")
else:
    print("⚠️ No candidates generated")

## 🧠 Agent 3: Reasoning Agent Output

The **Reasoning Agent** scored each facility using multiple criteria and ranked them.

In [ ]:
# Display Reasoning Agent output
if 'scored_facilities' in result:
    scored = result['scored_facilities']
    count = len(scored)
    print(f"🧠 SCORED & RANKED {count} FACILITIES")
    print("="*60)
    
    if count > 0:
        print("\n🏆 Top Ranked Facilities:\n")
        for i, facility in enumerate(scored[:3], 1):  # Show top 3
            score = facility.get('accessibility_score', 0)
            reasoning = facility.get('reasoning', 'No reasoning provided')
            
            print(f"{i}. {facility['name']}")
            print(f"   ⭐ Score: {score}/10")
            print(f"   📍 {facility.get('subcounty', 'N/A')}, {facility.get('ward', 'N/A')}")
            print(f"   💡 Why: {reasoning[:150]}...")
            print()
        
        print(f"💡 Key Achievement: Multi-factor scoring (accessibility + relevance + affordability)")
        print(f"⏱️  Time taken: {result.get('reasoning_time_seconds', 0):.2f}s")
    else:
        print("⚠️ No facilities scored")
else:
    print("⚠️ No scoring data available")

## 📝 Agent 4: Recommendation Agent Output

The **Recommendation Agent** created a complete, personalized service plan for Sarah.

In [ ]:
# Display the final recommendation
if 'recommendation' in result and result['recommendation']:
    recommendation = result['recommendation']
    print("📋 PERSONALIZED SERVICE PLAN")
    print("="*60)
    print(recommendation)
    print("\n" + "="*60)
    print(f"⏱️  Recommendation time: {result.get('recommendation_time_seconds', 0):.2f}s")
    print(f"\n💡 Key Achievement: Human-readable plan with actionable guidance")
else:
    print("⚠️ No recommendation generated")

## 💾 Memory Demonstration

One of the system's key features is **memory persistence**. Let's see how it remembers Sarah's profile and service plan.

In [ ]:
# Retrieve saved data from memory
session_summary = orchestrator.get_session_summary(user_id)

print("💾 MEMORY CONTENTS")
print("="*60)
print(f"👤 User ID: {user_id}")
print(f"🆔 Session ID: {session_summary.get('session_id', 'N/A')}")
print(f"📅 Created: {session_summary.get('session_created', 'N/A')}")
print(f"\n📊 Stored Data:")
print(f"   ✅ User Profile: {'Saved' if session_summary.get('user_profile') else 'Not saved'}")
print(f"   ✅ Service Plan: {'Saved' if session_summary.get('service_plan') else 'Not saved'}")
print(f"   ✅ Conversation History: {len(session_summary.get('conversation_history', []))} messages")

print(f"\n💡 Key Benefit: If Sarah returns later, we remember her profile and history!")
print("   - No need to re-ask disability type or location")
print("   - Can show her previous recommendations")
print("   - Can track progress (e.g., 'Did you visit Embakasi Health Center?')")

## 📊 Performance Metrics & Debug View

Let's examine the system's performance and see how long each agent took.

In [ ]:
# Create performance visualization
import pandas as pd

timing_data = [
    {'Agent': '1. Intake', 'Time (s)': result.get('intake_time_seconds', 0), 'Task': 'Extract user needs'},
    {'Agent': '2. Search', 'Time (s)': result.get('search_time_seconds', 0), 'Task': 'Find facilities'},
    {'Agent': '3. Reasoning', 'Time (s)': result.get('reasoning_time_seconds', 0), 'Task': 'Score & rank'},
    {'Agent': '4. Recommendation', 'Time (s)': result.get('recommendation_time_seconds', 0), 'Task': 'Create plan'},
]

timing_df = pd.DataFrame(timing_data)
total_time = result.get('total_time_seconds', timing_df['Time (s)'].sum())

print("⏱️  PERFORMANCE BREAKDOWN")
print("="*60)
print(timing_df.to_string(index=False))
print("="*60)
print(f"⚡ Total Pipeline Time: {total_time:.2f} seconds")
print(f"📊 Average per Agent: {total_time/4:.2f} seconds")

# Calculate data flow
print(f"\n📈 DATA FLOW STATISTICS")
print("="*60)
print(f"   Input: Natural language query ({len(demo_query)} characters)")
print(f"   → Intake: Extracted profile with {len(result.get('user_profile', {}))} fields")
print(f"   → Search: Found {len(result.get('candidate_facilities', []))} candidates")
print(f"   → Reasoning: Scored {len(result.get('scored_facilities', []))} facilities")
print(f"   → Recommendation: Generated {len(result.get('recommendation', ''))} character plan")
print(f"\n💡 Efficiency: {len(all_facilities)} facilities → {len(result.get('scored_facilities', []))} ranked in {total_time:.1f}s")

## 🎯 Key Insights & System Capabilities

### What Makes This System Unique?

**1. Multi-Agent Specialization** 🤖
- Each agent has ONE clear responsibility
- Easier to debug and improve individual components
- Can swap out agents without affecting others

**2. Structured Data Handoffs** 📋
- All inter-agent communication uses Pydantic models
- Type safety prevents errors
- Easy to validate data at each step

**3. Memory & Personalization** 💾
- Remembers user profiles across sessions
- Tracks conversation history
- Stores past service plans for follow-up

**4. Real-World Dataset** 🏥
- 25 real Nairobi facilities with detailed accessibility info
- Covers 10+ subcounties
- Realistic pricing and service categories

**5. Observable & Debuggable** 🔍
- Structured logging at every step
- Timing metrics for performance optimization
- Can inspect intermediate outputs

### Limitations & Future Improvements

**Current Limitations:**
- ⚠️ Dataset size: 25 facilities (needs expansion)
- ⚠️ No real-time data updates
- ⚠️ English only (no Swahili support yet)
- ⚠️ Simple distance calculations (no route optimization)

**Potential Enhancements:**
- ✨ Web scraping for live facility data
- ✨ Integration with Google Maps API for accurate routing
- ✨ WhatsApp/SMS interface for accessibility
- ✨ Multi-language support (Swahili, Kikuyu)
- ✨ User feedback loop for facility ratings
- ✨ Integration with M-Pesa for appointment booking

---

## 🚀 Try It Yourself!

Want to test the system with different queries? Run the cell below with your own scenario!

In [ ]:
# Try your own query!
# Examples:
# - "I'm deaf and need NCPWD office in CBD with sign language support"
# - "Blind person needs accessible clinic in Westlands"
# - "Parent with wheelchair-using child needs clinic in Langata"

custom_query = "I have limited mobility and need a social service office in Kibra"
custom_user_id = "custom_test_user"

print(f"📝 Testing query: {custom_query}\n")
custom_result = await orchestrator.process_query(
    user_id=custom_user_id,
    query=custom_query
)

print(f"\n✅ Complete! Check the recommendation below:\n")
print("="*60)
print(custom_result.get('recommendation', 'No recommendation generated'))
print("="*60)

## 📚 Next Steps & Additional Resources

### Explore More Notebooks

1. **`01_dataset_creation.ipynb`** - Learn how we curated the Nairobi facilities dataset
2. **`02_agent_development.ipynb`** - Deep dive into building each agent with ADK
3. **`04_evaluation_analysis.ipynb`** - See how we evaluate system performance

### Run Locally

```bash
# Clone the repository
git clone <your-repo-url>
cd implementation

# Install dependencies
pip install -r requirements.txt

# Set up Google Cloud credentials (optional for deployment)
export GOOGLE_APPLICATION_CREDENTIALS="path/to/credentials.json"

# Run the interactive CLI demo
python src/demo.py

# Run the evaluation suite
python tests/evaluation/run_evaluation.py
```

### Architecture Details

**Key Files:**
- `src/agents/` - All 4 agent definitions
- `src/tools/` - Custom tools (DatasetSearchTool, WebEnrichmentTool)
- `src/models/` - Pydantic schemas for data validation
- `src/orchestrator.py` - Agent coordination logic
- `src/memory/` - Session and long-term memory management
- `config/` - Agent configuration and settings

### Learn More

- 📖 [Google ADK Documentation](https://cloud.google.com/agent-development-kit)
- 🎓 [5-Day AI Agents Intensive Course](https://www.deeplearning.ai)
- 🇰🇪 [Nairobi Disability Statistics](https://www.ncpwd.go.ke)
- ♿ [Web Accessibility Guidelines](https://www.w3.org/WAI/)

---

### 💬 Questions or Feedback?

If you have questions about this implementation or suggestions for improvements:
- Open an issue on GitHub
- Reach out to the author
- Contribute to the project!

**Thank you for exploring this project!** 🙏